In [ ]:
from pathlib import Path
import tifffile as tf
import numpy as np 
import histomicstk as htk
import matplotlib.pyplot as plt
import skimage
import cv2
import pandas as pd

In [ ]:
registration_base_path =  'Y:/coskun-lab/Aditi/01/Results/registration_algorithms_comparison'

In [ ]:
key = '/Multiplex 10/Med 1'

In [ ]:
complete_registration_path = registration_base_path + '/SIFT' + key

In [ ]:
HE_registered = tf.imread(complete_registration_path + '/HE_registered.tif')
bleached_HE_registered = tf.imread(complete_registration_path + '/bleached_HE_registered.tif')

In [ ]:
PAS_registered = tf.imread(complete_registration_path + '/PAS_registered.tif')
bleached_PAS_registered = tf.imread(complete_registration_path + '/bleached_PAS_registered.tif')

In [ ]:
HE_registered = np.clip(HE_registered * 255, 0, 255).astype(np.uint8)
bleached_HE_registered = np.clip(bleached_HE_registered * 255, 0, 255).astype(np.uint8)

In [ ]:
PAS_registered = np.clip(PAS_registered * 255, 0, 255).astype(np.uint8)
bleached_PAS_registered = np.clip(bleached_PAS_registered * 255, 0, 255).astype(np.uint8)

In [ ]:
stain_color_map = htk.preprocessing.color_deconvolution.stain_color_map
stains = ['hematoxylin', 'eosin','null']         
w_est_HE_given = np.array([stain_color_map[st] for st in stains]).T
HE_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(HE_registered, w_est_HE_given)
bleached_HE_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(bleached_HE_registered, w_est_HE_given)

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_figheight(15)
fig.set_figwidth(15)
ax[0].imshow(HE_registered[5000:6000, 9500:10500])
ax[1].imshow(bleached_HE_registered[5000:6000, 9500:10500])

In [ ]:
w_est_PAS = htk.preprocessing.color_deconvolution.rgb_separate_stains_macenko_pca(PAS_registered[5000:6000, 10000:11000], 255)
PAS_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(PAS_registered, w_est_PAS)
bleached_PAS_deconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(bleached_PAS_registered, w_est_PAS)

In [ ]:
eosin  = HE_deconvolved.Stains[:, :, 1]
eosin_bleached = bleached_HE_deconvolved.Stains[:, :, 1]

In [ ]:
pas  = PAS_deconvolved.Stains[:, :, 2]
pas_bleached = bleached_PAS_deconvolved.Stains[:, :, 2]

In [ ]:
h, w = HE_registered.shape[:2]
tile_size = 1024

eosin_intensity_sum = []
eosin_bleached_intensity_sum = []

pas_intensity_sum = []
pas_bleached_intensity_sum = []

for y in range(0, h - tile_size + 1, tile_size):
    for x in range(0, w - tile_size + 1, tile_size):
        eosin_intensity_sum.append(np.sum(255-eosin[y:y+tile_size, x:x+tile_size]))
        eosin_bleached_intensity_sum.append(np.sum(255-eosin_bleached[y:y+tile_size, x:x+tile_size]))
        pas_intensity_sum.append(np.sum(255-pas[y:y+tile_size, x:x+tile_size]))
        pas_bleached_intensity_sum.append(np.sum(255-pas_bleached[y:y+tile_size, x:x+tile_size]))


In [ ]:
len(eosin_intensity_sum)

In [ ]:
import seaborn as sns
from statannotations.Annotator import Annotator
import matplotlib as mpl

mpl.rcParams['font.size']=12
df = pd.DataFrame({
    "Stain": (
        ['Eosin'] * len(eosin_intensity_sum) +
        ['Eosin Bleach'] * len(eosin_bleached_intensity_sum)
    ),
    "value": eosin_intensity_sum + eosin_bleached_intensity_sum
})

plt.figure(figsize=(3, 4))
ax = sns.barplot(
    data=df,
    x="Stain",
    y="value",
    errorbar="se",
    capsize=0.2, 
    palette = ["hotpink", "grey"], 
    width=0.5
)

# Statistical comparisons (edit pairs as needed)
pairs = [
    ("Eosin", "Eosin Bleach")
]

annotator = Annotator(ax, pairs, data=df, x="Stain", y="value")
annotator.configure(
    test="t-test_paired",
    text_format="star",
    loc="outside"
)
annotator.apply_and_annotate()

ax.set_ylabel("Intensity Sum")
ax.set_xlabel("Stain")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib as mpl
mpl.rcParams['font.size']=12

df = pd.DataFrame({
    "group": (
        ['PAS'] * len(pas_intensity_sum) +
        ['PAS Bleach'] * len(pas_bleached_intensity_sum)
    ),
    "value": pas_intensity_sum + pas_bleached_intensity_sum
})

plt.figure(figsize=(3, 4))
ax = sns.barplot(
    data=df,
    x="group",
    y="value",
    errorbar="se",
    capsize=0.2, 
    palette=['mediumorchid', 'grey'], 
    width=0.5
)

# Statistical comparisons (edit pairs as needed)
pairs = [
    ("PAS", "PAS Bleach")
]

annotator = Annotator(ax, pairs, data=df, x="group", y="value")
annotator.configure(
    test="t-test_paired",
    text_format="star",
    loc="outside"
)
annotator.apply_and_annotate()

ax.set_ylabel("Intensity Sum")
ax.set_xlabel("Stain")
plt.tight_layout()
plt.show()
